In [3]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ================= 1. 数据加载与预处理 =================
file_path = '智能音箱数据集.xlsx'
df = pd.read_excel(file_path)

# 去除所有列名前后的隐藏空格
df.columns = df.columns.str.strip()

# 建立灵活的列名映射字典（已补充你数据集中的真实表头）
column_mapping = {
    'func_col': ['function_name', '功能名称', '指令名称', '执行功能', '功能调用类型'],  # 【新增】
    'time_col': ['response_time_ms', '响应时间', '响应时长(ms)', '耗时']              # 【新增】
}

def find_real_column(df, candidates):
    """从候选列表中找出数据集中实际存在的列名"""
    for col in candidates:
        if col in df.columns:
            return col
    raise KeyError(f"❌ 找不到匹配的列！请检查Excel表头是否包含以下任意一项: {candidates}")

# 动态获取真实的列名
real_func_col = find_real_column(df, column_mapping['func_col'])
real_time_col = find_real_column(df, column_mapping['time_col'])

print(f"✅ 成功匹配到功能列: '{real_func_col}'")
print(f"✅ 成功匹配到时间列: '{real_time_col}'")
print("=" * 60)

# ================= 2. M1 & M2：用户使用习惯与频率分析 =================
print("\n【M1 & M2】功能使用频率分析")

# 统计各功能的调用次数并按降序排列
func_counts = df[real_func_col].value_counts()

# M1: 提取使用最频繁的前三个功能
top_3_funcs = func_counts.head(3).index.tolist()
print(f"🔥 使用最频繁的三个功能为: {', '.join(top_3_funcs)}")

# M2: 识别最受欢迎和最低频的功能
most_popular = func_counts.idxmax()
least_popular = func_counts.idxmin()
print(f"⭐ 最受欢迎的功能: {most_popular}")
print(f"📉 使用频率较低的功能: {least_popular}")

# ================= 3. M3：响应时间分析 =================
print("\n【M3】响应时间分析")

# 确保时间列为数值类型，防止字符串或单位导致计算报错
df[real_time_col] = pd.to_numeric(df[real_time_col], errors='coerce')

# 按功能分组，计算平均响应时间
avg_response_time = df.groupby(real_func_col)[real_time_col].mean().sort_values(ascending=False)

# 找出响应时间最长、最短及适中的功能
slowest_func = avg_response_time.index[0]
fastest_func = avg_response_time.index[-1]
moderate_funcs = avg_response_time.iloc[1:-1].index.tolist()

print(f"🐢 响应时间最长的功能: {slowest_func} ({avg_response_time.iloc[0]:.2f} ms)")
print(f"⚡ 响应时间最短的功能: {fastest_func} ({avg_response_time.iloc[-1]:.2f} ms)")
print(f"⏱️ 响应时间适中的功能: {', '.join(moderate_funcs)}")


✅ 成功匹配到功能列: '功能调用类型'
✅ 成功匹配到时间列: '响应时间'

【M1 & M2】功能使用频率分析
🔥 使用最频繁的三个功能为: 调整音量, 查询新闻, 查天气
⭐ 最受欢迎的功能: 调整音量
📉 使用频率较低的功能: 播放音乐

【M3】响应时间分析
🐢 响应时间最长的功能: 控制家居 (2.07 ms)
⚡ 响应时间最短的功能: 查询新闻 (1.87 ms)
⏱️ 响应时间适中的功能: 查天气, 播放音乐, 设置闹钟, 查询知识, 调整音量, 提醒事项
